# Day 18 · Colab 1 — Claude Agent + FastAPI Backend + Redis Memory

**Agentic Systems Bootcamp**

In this notebook you build a small but complete **tool-using agent**:

- a **FastAPI** backend exposing an `/orders/{id}` endpoint (your *external API*),
- a **Redis** memory layer (short-term conversation history + long-term facts),
- a **Claude** agent that calls tools in a `while` loop driven by `stop_reason`.

Everything runs **inside Colab** with no external servers:
`fakeredis` stands in for Redis and FastAPI's `TestClient` calls the API in-process.
Swap either for the real thing by changing one line each (shown at the end).

> **Pattern recap (from the deck):** client tools return `stop_reason="tool_use"`; *your* code runs the
> tool and returns a `tool_result` block; you loop until `stop_reason="end_turn"`.

## Step 1 — Install dependencies & set your API key

`fakeredis` gives us a real Redis API surface in-process. `httpx` is needed by FastAPI's test client.

In [ ]:
!pip install -q anthropic fastapi 'httpx<0.28' fakeredis 2>/dev/null
print('deps installed')

In [ ]:
import os, getpass

# In Colab, prefer the Secrets panel (key icon) named ANTHROPIC_API_KEY.
if not os.environ.get('ANTHROPIC_API_KEY'):
    try:
        from google.colab import userdata
        os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
    except Exception:
        pass

if not os.environ.get('ANTHROPIC_API_KEY'):
    # Fallback: paste it (hidden). Leave blank to run in OFFLINE mock mode.
    os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('ANTHROPIC_API_KEY (blank = offline mock): ')

LIVE = bool(os.environ.get('ANTHROPIC_API_KEY'))
MODEL = 'claude-sonnet-4-6'  # fast + cheap for a workshop; opus-4-8 for harder reasoning
print('LIVE mode' if LIVE else 'OFFLINE mock mode (no key) — agent loop will be simulated')

## Step 2 — A tiny FastAPI backend (the 'external API')

This is the kind of service your agent does **not** control — it just calls it.
We expose one route, then wrap the app in a `TestClient` so calls run in-process.

In [ ]:
from fastapi import FastAPI, HTTPException
from fastapi.testclient import TestClient

app = FastAPI()

_ORDERS = {
    'A1001': {'id': 'A1001', 'item': 'Mechanical keyboard', 'qty': 1, 'status': 'shipped',  'total': 129.0},
    'A1002': {'id': 'A1002', 'item': 'USB-C hub',          'qty': 2, 'status': 'processing','total': 58.0},
    'A1003': {'id': 'A1003', 'item': '4K monitor',         'qty': 1, 'status': 'delivered', 'total': 410.0},
}

@app.get('/orders/{order_id}')
def get_order(order_id: str):
    o = _ORDERS.get(order_id.upper())
    if not o:
        raise HTTPException(status_code=404, detail='order not found')
    return o

client = TestClient(app)
print(client.get('/orders/A1001').json())

## Step 3 — A Redis memory layer

Two responsibilities, the way the deck framed memory:

- **Short-term** — the running conversation, stored as a Redis **list** per session (`hist:<session>`).
- **Long-term** — durable **facts** about the user, stored in a Redis **hash** (`facts:<session>`), with an optional TTL.

`fakeredis` implements the same commands, so this class is unchanged when you move to real Redis.

In [ ]:
import json, time, fakeredis

class RedisMemory:
    def __init__(self, r, session_id: str, history_limit: int = 40):
        self.r = r
        self.sid = session_id
        self.history_limit = history_limit
        self.h_key = f'hist:{session_id}'
        self.f_key = f'facts:{session_id}'

    # ---- short-term: conversation turns ----
    def append_turn(self, role: str, content):
        self.r.rpush(self.h_key, json.dumps({'role': role, 'content': content}))
        self.r.ltrim(self.h_key, -self.history_limit, -1)  # keep only the tail

    def load_history(self):
        return [json.loads(x) for x in self.r.lrange(self.h_key, 0, -1)]

    # ---- long-term: durable facts ----
    def set_fact(self, key: str, value: str, ttl_seconds: int | None = None):
        self.r.hset(self.f_key, key, value)
        if ttl_seconds:
            self.r.expire(self.f_key, ttl_seconds)

    def get_fact(self, key: str):
        v = self.r.hget(self.f_key, key)
        return v.decode() if isinstance(v, bytes) else v

    def all_facts(self):
        return {k.decode(): v.decode() for k, v in self.r.hgetall(self.f_key).items()}

r = fakeredis.FakeStrictRedis()
mem = RedisMemory(r, session_id='demo-user')
mem.set_fact('name', 'Asha')
mem.append_turn('user', 'hello')
print('facts:', mem.all_facts())
print('history:', mem.load_history())

## Step 4 — Declare the tools (JSON schemas)

Three client tools. Names are **namespaced by purpose** and every field is described —
the description *is* the prompt the model reads when deciding how to call.
`order_id` uses a `pattern` so the model returns well-formed IDs.

In [ ]:
TOOLS = [
    {
        'name': 'get_order',
        'description': 'Look up a customer order by its ID and return item, quantity, status and total.',
        'input_schema': {
            'type': 'object',
            'properties': {
                'order_id': {'type': 'string', 'description': "Order ID like 'A1001'.", 'pattern': '^[Aa][0-9]{4}$'}
            },
            'required': ['order_id'],
        },
    },
    {
        'name': 'remember_fact',
        'description': 'Persist a durable fact about the user (e.g. shipping preference) for future turns.',
        'input_schema': {
            'type': 'object',
            'properties': {
                'key':   {'type': 'string', 'description': 'Short fact key, e.g. "shipping_pref".'},
                'value': {'type': 'string', 'description': 'The fact value to store.'},
            },
            'required': ['key', 'value'],
        },
    },
    {
        'name': 'recall_fact',
        'description': 'Retrieve a previously stored fact about the user by key. Returns empty if unknown.',
        'input_schema': {
            'type': 'object',
            'properties': {'key': {'type': 'string', 'description': 'The fact key to look up.'}},
            'required': ['key'],
        },
    },
]
print(len(TOOLS), 'tools declared')

## Step 5 — A dispatch map: tool name → Python function

This is the boundary between *the model's intent* and *your code*. Each function returns a
string (what the model will read back). Errors are **returned**, not raised, so the agent can recover —
that maps to `is_error: true` on the `tool_result` block in Step 6.

In [ ]:
def tool_get_order(order_id: str):
    resp = client.get(f'/orders/{order_id}')
    if resp.status_code == 404:
        return {'error': f'No order {order_id} found.'}
    resp.raise_for_status()
    return resp.json()

def tool_remember_fact(key: str, value: str):
    mem.set_fact(key, value)
    return {'ok': True, 'stored': {key: value}}

def tool_recall_fact(key: str):
    v = mem.get_fact(key)
    return {'key': key, 'value': v} if v is not None else {'key': key, 'value': None}

DISPATCH = {
    'get_order': tool_get_order,
    'remember_fact': tool_remember_fact,
    'recall_fact': tool_recall_fact,
}

def run_tool(name, args):
    fn = DISPATCH.get(name)
    if fn is None:
        return {'error': f'unknown tool {name}'}, True
    try:
        out = fn(**args)
        is_err = isinstance(out, dict) and 'error' in out
        return out, is_err
    except Exception as e:
        return {'error': repr(e)}, True

print(run_tool('get_order', {'order_id': 'A1002'}))
print(run_tool('get_order', {'order_id': 'A9999'}))

## Step 6 — The agent loop (driven by `stop_reason`)

This is the heart of client-side tool use:

1. send messages + tool definitions,
2. if `stop_reason == 'tool_use'`: run **every** `tool_use` block, append a single user message
   containing one `tool_result` per call (matched by `tool_use_id`), and loop,
3. stop at `stop_reason == 'end_turn'` and return the text.

If there's no API key we simulate one scripted tool call so the notebook still runs end-to-end.

In [ ]:
import json

SYSTEM = (
    'You are an order-support assistant. Use get_order for any order question. '
    'Use remember_fact / recall_fact to keep durable user preferences across turns. '
    'Be concise.'
)

def agent_turn(user_text, max_steps=6, verbose=True):
    mem.append_turn('user', user_text)
    messages = mem.load_history()

    if not LIVE:
        # ---- offline mock: pretend the model asked for get_order once ----
        if verbose: print('… (mock) model requests get_order A1001')
        out, _ = run_tool('get_order', {'order_id': 'A1001'})
        reply = f'(mock) Order A1001 is {out.get("status", "?")}.'
        mem.append_turn('assistant', reply)
        return reply

    from anthropic import Anthropic
    clientA = Anthropic()
    for step in range(max_steps):
        resp = clientA.messages.create(
            model=MODEL, max_tokens=1024, system=SYSTEM, tools=TOOLS, messages=messages,
        )
        if resp.stop_reason == 'tool_use':
            # echo the assistant turn (text + tool_use blocks) back into the transcript
            messages.append({'role': 'assistant', 'content': [b.model_dump() for b in resp.content]})
            results = []
            for block in resp.content:
                if block.type == 'tool_use':
                    if verbose: print(f'  → tool: {block.name}({block.input})')
                    out, is_err = run_tool(block.name, block.input)
                    results.append({
                        'type': 'tool_result',
                        'tool_use_id': block.id,
                        'content': json.dumps(out),
                        'is_error': is_err,
                    })
            messages.append({'role': 'user', 'content': results})
            continue
        # end_turn
        text = ''.join(b.text for b in resp.content if b.type == 'text')
        mem.append_turn('assistant', text)
        return text
    return '(stopped: max steps reached)'

print(agent_turn('What is the status of order A1002?'))

## Step 7 — Persistence check: memory survives across turns

Because every turn is written to Redis, a fresh `agent_turn` call still sees the history and facts.
Ask the agent to remember something, then recall it in a later turn.

In [ ]:
print(agent_turn('Please remember that my shipping preference is express.'))
print('---')
print(agent_turn('What did I say my shipping preference was?'))
print('---')
print('Raw facts in Redis:', mem.all_facts())
print('History length:', len(mem.load_history()), 'turns')

## Step 8 — Multi-turn chat demo

A short scripted conversation that exercises lookup + memory together.

In [ ]:
for msg in [
    'Hi, I am Asha.',
    'How much was order A1003?',
    'Remember that my budget cap is 500 dollars.',
    'Given my budget cap, was that order within it?',
]:
    print('USER:', msg)
    print('AGENT:', agent_turn(msg, verbose=False))
    print()

## Extension tasks

Pick a few — these are the assignment for this lab.

1. **Rolling summary / compaction.** When `load_history()` exceeds N turns, summarise the oldest
   turns with a cheap model call and replace them with one synthetic `assistant` summary message.
   Keep total context bounded.
2. **TTL & a `forget` tool.** Add a `forget_fact(key)` tool and give facts a TTL via `set_fact(..., ttl_seconds=...)`.
   Show that an expired fact returns `None`.
3. **Parallel lookups.** Add a `get_customer` tool and a second order endpoint, then ask a question that
   needs both — observe Claude emit **two `tool_use` blocks in one turn**. Confirm your loop answers both ids.
4. **Guarded writes / PII.** Reject `remember_fact` values that look like card numbers or emails (regex);
   return `is_error: true` with a helpful message and confirm the agent recovers.
5. **Token accounting.** Read `resp.usage` each step; print cumulative input/output tokens for a turn.
6. **Real Redis.** Replace `fakeredis.FakeStrictRedis()` with `redis.Redis.from_url(os.environ['REDIS_URL'])`
   (e.g. a free Upstash/Redis Cloud URL). The `RedisMemory` class should not change at all.

> **Stretch:** swap `TestClient` for a real deployed FastAPI URL using `httpx.Client(base_url=...)`.
> Only `tool_get_order` changes; the agent loop is untouched.